# VAE Introspection Demo

This notebook demonstrates how to visualize and understand what's happening inside a VAE.

## What We'll Learn

1. **Activation Maps**: What features do the encoder's conv layers detect?
2. **Latent Traversals**: What does each dimension of z control?
3. **Ablation Studies**: What information is in z_c vs z_d?
4. **Saliency Maps**: Which input pixels affect which latent dimensions?
5. **Dimension Importance**: Which latent dimensions carry the most information?

In [ ]:
# First, run the training sandbox to get a trained model
# (or load a pre-trained one)

%run sepvae_training_sandbox.ipynb

In [ ]:
# Import introspection toolkit
from vae_introspection import VAEIntrospector, quick_introspect

# Create introspector
introspector = VAEIntrospector(model, device=str(DEVICE))

---
## 1. Get Sample Images for Analysis

In [ ]:
# Get one sample from each class
samples = {}
for x, labels in test_loader:
    for i in range(len(x)):
        label = labels[i].item()
        if label not in samples:
            samples[label] = (x[i:i+1], labels[i:i+1])
    if len(samples) == len(class_names):
        break

# Show samples
fig, axes = plt.subplots(1, len(samples), figsize=(4*len(samples), 4))
for idx, (label, (x, _)) in enumerate(samples.items()):
    axes[idx].imshow(x.squeeze().numpy() * 0.5 + 0.5, cmap='gray')
    axes[idx].set_title(f'{class_names[label]} (label={label})')
    axes[idx].axis('off')
plt.suptitle('Test Samples for Introspection', fontsize=14)
plt.tight_layout()
plt.show()

# Select a disease sample for analysis
x_disease, label_disease = samples[1]  # Disease_1
x_normal, label_normal = samples[0]    # Normal

---
## 2. Encoder Activation Maps

**What are we looking for?**
- **Early layers**: Edge detectors, texture patterns
- **Middle layers**: Part detectors, local structure
- **Late layers**: Semantic features, global patterns

Each channel in a layer acts as a **feature detector**. The activation map shows where in the image that feature was detected.

In [ ]:
# Visualize encoder activations for a disease sample
fig = introspector.visualize_encoder_activations(x_disease, label_disease)
plt.show()

In [ ]:
# Compare with normal sample
fig = introspector.visualize_encoder_activations(x_normal, label_normal)
plt.show()

### Interpretation Guide

| Layer | Expected Pattern | What It Means |
|-------|-----------------|---------------|
| Conv1 (14×14×32) | Edges, gradients | Low-level features |
| Conv2 (7×7×64) | Corners, textures | Mid-level combinations |
| Conv3 (4×4×128) | Object parts, shapes | High-level semantics |

**If activations look similar across classes**: The encoder may not be learning discriminative features.

**If some channels are always zero**: Those filters are "dead" and not learning.

---
## 3. Decoder Activation Maps

Shows how the decoder builds up the reconstruction from the latent code.
- **Early decoder layers**: Coarse structure, overall shape
- **Late decoder layers**: Fine details, textures

In [ ]:
fig = introspector.visualize_decoder_activations(x_disease, label_disease)
plt.show()

---
## 4. Latent Traversals: What Does Each Dimension Control?

This is the **key visualization** for understanding latent representations.

For each dimension:
- We fix all other dimensions at their encoded values
- We vary the target dimension from -3σ to +3σ
- We observe what changes in the reconstruction

**Ideal behavior:**
- Each dimension should control one interpretable factor
- Traversals should be smooth (no sudden jumps)
- z_c dimensions should change anatomy, not disease features
- z_d dimensions should change disease features, not anatomy

In [ ]:
# Traverse COMMON latent dimensions (z_c)
# These should encode class-invariant features (anatomy)
fig = introspector.latent_traversal(x_disease, label_disease, latent_type='common', n_dims=8)
plt.show()

In [ ]:
# Traverse DISEASE latent dimensions (z_d)
# These should encode disease-specific features
fig = introspector.latent_traversal(x_disease, label_disease, latent_type='disease', n_dims=8)
plt.show()

### Interpretation Guide

| Observation | Diagnosis |
|-------------|----------|
| Smooth transitions | Good manifold coverage |
| Sudden jumps/artifacts | Manifold has voids |
| No change when varying | Dimension not used (collapse) |
| Multiple things change | Entangled representation |
| z_c changes disease features | Disease leaking into common (MI too weak) |
| z_d changes anatomy | Separation not working |

In [ ]:
# 2D traversal grid - vary two dimensions simultaneously
# Reveals interactions between dimensions
fig = introspector.latent_traversal_grid(
    x_disease, label_disease, 
    dim1=0, dim2=1,  # Try different dimension pairs
    latent_type='common',
    n_steps=7
)
plt.show()

---
## 5. Ablation Study: What Information is in z_c vs z_d?

We systematically zero out different parts of the latent code to see what information each part carries.

In [ ]:
fig = introspector.ablation_study(x_disease, label_disease)
plt.show()

### What to Look For

| Condition | Expected Result | If Not Working |
|-----------|-----------------|----------------|
| z_c only | Generic shape, no disease features | Disease leaking into z_c |
| z_d only | Disease-specific pattern only | z_d encoding too much |
| Random z_c, keep z_d | Different anatomy, same disease | Good separation |
| Keep z_c, random z_d | Same anatomy, different disease | Good separation |

---
## 6. Dimension Importance: Which Dimensions Matter Most?

We measure importance by seeing how much reconstruction error increases when each dimension is zeroed out.

In [ ]:
# Common latent dimension importance
fig = introspector.dimension_importance(x_disease, label_disease, latent_type='common')
plt.show()

In [ ]:
# Disease latent dimension importance
fig = introspector.dimension_importance(x_disease, label_disease, latent_type='disease')
plt.show()

### Interpretation

- **High importance (tall bars)**: Dimension carries critical information
- **Near-zero importance**: Dimension may be redundant or collapsed
- **Uneven distribution**: Some dimensions are overloaded

**Ideal**: Relatively even distribution of importance across dimensions

---
## 7. Saliency Analysis: Which Pixels Affect Which Latents?

Uses gradients to show which input pixels have the most influence on each latent dimension.

**Key insight**: This reveals what the encoder "looks at" when computing each latent dimension.

In [ ]:
fig = introspector.full_saliency_analysis(x_disease, label_disease, n_dims=8)
plt.show()

In [ ]:
# Detailed saliency for a specific dimension
fig = introspector.latent_saliency_map(x_disease, label_disease, latent_type='disease', dim=0)
plt.show()

### What Good Saliency Maps Look Like

| Latent Type | Expected Saliency Pattern |
|-------------|---------------------------|
| z_c dimensions | Global/structural regions |
| z_d dimensions | Disease-specific regions |

**If z_d saliency is everywhere**: Disease features not localized

**If z_c and z_d saliency overlap heavily**: Poor disentanglement

---
## 8. Interpolation with Activation Tracking

Interpolate between two samples while monitoring internal activations.

Reveals:
- Smoothness of the manifold
- How activations change continuously (or not)

In [ ]:
# Interpolate from normal to disease
fig = introspector.latent_interpolation_with_activations(
    x_normal, x_disease,
    label_normal, label_disease,
    n_steps=10
)
plt.show()

---
## 9. Compare Normal vs Disease Encodings

In [ ]:
# Encode both samples
model.eval()
with torch.no_grad():
    z_c_normal, z_d_normal, _ = model.encode(x_normal.to(DEVICE), label_normal.to(DEVICE))
    z_c_disease, z_d_disease, _ = model.encode(x_disease.to(DEVICE), label_disease.to(DEVICE))

# Compare latent distributions
fig, axes = plt.subplots(2, 2, figsize=(14, 10))

# z_c comparison
z_c_n = z_c_normal[0].cpu().numpy()
z_c_d = z_c_disease[0].cpu().numpy()
x_dims = np.arange(len(z_c_n))

axes[0, 0].bar(x_dims - 0.2, z_c_n, 0.4, label='Normal', color='blue', alpha=0.7)
axes[0, 0].bar(x_dims + 0.2, z_c_d, 0.4, label='Disease', color='red', alpha=0.7)
axes[0, 0].set_xlabel('Dimension')
axes[0, 0].set_ylabel('Value')
axes[0, 0].set_title('z_c (Common Latent) Comparison')
axes[0, 0].legend()
axes[0, 0].axhline(0, color='k', linewidth=0.5)

# z_d comparison
z_d_n = z_d_normal[0].cpu().numpy()
z_d_d = z_d_disease[0].cpu().numpy()
x_dims_d = np.arange(len(z_d_n))

axes[0, 1].bar(x_dims_d - 0.2, z_d_n, 0.4, label='Normal', color='blue', alpha=0.7)
axes[0, 1].bar(x_dims_d + 0.2, z_d_d, 0.4, label='Disease', color='red', alpha=0.7)
axes[0, 1].set_xlabel('Dimension')
axes[0, 1].set_ylabel('Value')
axes[0, 1].set_title('z_d (Disease Latent) Comparison')
axes[0, 1].legend()
axes[0, 1].axhline(0, color='k', linewidth=0.5)

# Difference plot
diff_c = z_c_d - z_c_n
diff_d = z_d_d - z_d_n

axes[1, 0].bar(x_dims, diff_c, color='purple', alpha=0.7)
axes[1, 0].set_xlabel('Dimension')
axes[1, 0].set_ylabel('Difference')
axes[1, 0].set_title('z_c Difference (Disease - Normal)')
axes[1, 0].axhline(0, color='k', linewidth=0.5)

axes[1, 1].bar(x_dims_d, diff_d, color='orange', alpha=0.7)
axes[1, 1].set_xlabel('Dimension')
axes[1, 1].set_ylabel('Difference')
axes[1, 1].set_title('z_d Difference (Disease - Normal)')
axes[1, 1].axhline(0, color='k', linewidth=0.5)

plt.suptitle('Latent Code Comparison: Normal vs Disease', fontsize=14)
plt.tight_layout()
plt.show()

# Summary statistics
print("\n=== Latent Code Statistics ===")
print(f"z_c distance: {np.linalg.norm(diff_c):.4f}")
print(f"z_d distance: {np.linalg.norm(diff_d):.4f}")
print(f"\nz_c should be SIMILAR (small difference) if common features are class-invariant")
print(f"z_d should be DIFFERENT (large difference) for disease samples")

---
## 10. Run Full Introspection Report

In [ ]:
# Complete analysis for one sample
introspector.visualize_all(x_disease, label_disease)

---
## Summary: What Good Latent Representations Look Like

| Criterion | How to Check | Good Sign | Bad Sign |
|-----------|-------------|-----------|----------|
| **Smooth manifold** | Latent traversals | Gradual transitions | Sudden jumps |
| **Dimension usage** | Importance analysis | Even distribution | Most dims near zero |
| **Disentanglement** | Single-dim traversals | One factor changes | Multiple things change |
| **z_c/z_d separation** | Ablation study | z_c = anatomy, z_d = disease | Both encode everything |
| **Meaningful features** | Saliency maps | Focus on relevant regions | Random/uniform saliency |
| **Interpolation** | Activation tracking | Smooth activation changes | Discontinuities |